# Bandit Experiments

Example notebook demonstrating usage of the ibrl library in some basic experiments. 

Shows how to use existing/pre-specified agents and environments, including:
- **Classical agents**: Q-learning, Bayesian, EXP3, Experimental2
- **UCB**: Classic Upper Confidence Bound (deterministic, one-hot policies)
- **IUCB**: Imprecise UCB (Kosoy 2024) — cycle-based algorithm with confidence set geometry, designed for Newcomb-like games

Note that it is possible to define your own agents and/or environments and run similar experiments.

In [ ]:
import numpy as np

from ibrl.simulators import simulate
from ibrl.utils import construct_environment, construct_agent

## Configuration

Edit the lists below to choose which environments and agents to run.

In [ ]:
# EG, some currently available environments (may change):
#   "bandit", "switching", "newcomb", "damascus",
#   "asymmetric-damascus", "coordination", "pdbandit"
# You can also pass options, e.g. "newcomb:transparency=0.9"

environments = [
    "bandit",
    "newcomb",
    "switching",
    "damascus",
    #"asymmetric-damascus", 
    #"coordination", 
    #"pdbandit",
]

# EG, some currently available agents (may change):
#   "classical", "bayesian", "exp3", "experimental1", "experimental2",
#   "ucb", "iucb"
# You can also pass options, e.g. "classical:epsilon=0.1", "ucb:exploration=1.0"

agents = [
    "classical",
    "bayesian",
    "exp3",
    "experimental2",
    "ucb",
    "iucb",
]

# Shared options
options = {
    "num_actions": 2,
    "num_steps":   500,
    "num_runs":    25,
    "seed":        42,
    "verbose":     0,
}

## Run experiments

In [ ]:
all_results = {}

for env_name in environments:
    all_results[env_name] = {}
    for agent_name in agents:
        label = f"{env_name} / {agent_name}"
        print(f"Running: {label}")

        env = construct_environment(env_name, options)
        agent = construct_agent(agent_name, options)
        results = simulate(env, agent, options)
        all_results[env_name][agent_name] = results

        print(f"  optimal reward: {results['optimal_reward']:.4f}")
        print(f"  final avg reward: {results['average_reward'][0, -1]:.4f}")

print("Done.")

## Plot results

In [ ]:
from ibrl.environments import BanditEnvironment
bandit = BanditEnvironment(num_actions=2)

In [ ]:
import math
import matplotlib.pyplot as plt

cols = 3

for env_name, agent_results in all_results.items():
    num_agents = len(agent_results)
    num_rows = math.ceil(num_agents / cols) + 1  # +1 for the regret row
    k_val = options["num_actions"]

    fig = plt.figure(figsize=(cols * 4, num_rows * 4))
    fig.suptitle(f"Environment: {env_name}", fontsize=18, fontweight='bold')

    # Top plot: Cumulative Regret (spans first row)
    ax_regret = plt.subplot(num_rows, 1, 1)
    for name, res in agent_results.items():
        avg_reward = res["average_reward"][0, :]
        optimal = res["optimal_reward"]
        cumulative_regret = np.cumsum(optimal - avg_reward)
        ax_regret.plot(cumulative_regret, label=name, linewidth=2)
    ax_regret.set_title("Cumulative Regret")
    ax_regret.legend(loc='upper left', bbox_to_anchor=(1, 1))
    ax_regret.grid(True, alpha=0.3)

    # Bottom subplots: Selection Probabilities per agent
    for i, (name, res) in enumerate(agent_results.items()):
        ax_act = plt.subplot(num_rows, cols, i + cols + 1)
        actions = res["actions"]  # (num_runs, num_steps, num_actions)
        for arm in range(k_val):
            freq = (actions == arm).mean(axis=0)
            ax_act.plot(freq, label=f"Arm {arm}")

        ax_act.set_title(f"{name}")
        ax_act.set_ylim(-0.05, 1.05)
        if i % cols == 0:
            ax_act.set_ylabel("Selection Prob")
        if i == num_agents - 1:
            ax_act.legend(loc='center left', bbox_to_anchor=(1.05, 0.5))

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

## UCB vs IUCB: Comparison on Newcomb-like games

UCB plays deterministic (one-hot) policies. On standard bandits this is optimal, but on Newcomb-like environments the predictor can perfectly match a deterministic agent.

IUCB (Imprecise UCB) uses cycle-based updates and a confidence set over payoff matrices. It models the predictor as adversarial (minimax), which allows it to discover that mixed strategies are needed on games like Damascus.

Key predictions:
- **Bandit**: Both UCB and IUCB should achieve low regret
- **Damascus**: UCB gets reward 0 (pure strategy → predictor always matches → death). IUCB discovers the p=0.5 mixed equilibrium (value 5)
- **Newcomb**: UCB one-boxes (deterministic policy sees diagonal entries: 10 > 5). IUCB two-boxes (minimax/CDT answer: game value 5)

In [ ]:
# Summary table: final average reward (last 50 steps) for each agent/environment
focus_envs = ["bandit", "newcomb", "damascus"]
focus_agents = ["ucb", "iucb", "classical", "experimental2"]

print(f"{'Agent':<16}", end="")
for env_name in focus_envs:
    print(f"  {env_name:>12}", end="")
print()
print("-" * 56)

for agent_name in focus_agents:
    print(f"{agent_name:<16}", end="")
    for env_name in focus_envs:
        if env_name in all_results and agent_name in all_results[env_name]:
            res = all_results[env_name][agent_name]
            final_avg = res["average_reward"][0, -50:].mean()
            print(f"  {final_avg:>12.2f}", end="")
        else:
            print(f"  {'N/A':>12}", end="")
    print()

print("-" * 56)
print(f"{'optimal':<16}", end="")
for env_name in focus_envs:
    if env_name in all_results:
        first_agent = next(iter(all_results[env_name]))
        opt = all_results[env_name][first_agent]["optimal_reward"]
        print(f"  {opt:>12.2f}", end="")
print()

In [ ]:
# UCB vs IUCB: policy evolution on Damascus
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, agent_name in zip(axes, ["ucb", "iucb"]):
    if "damascus" in all_results and agent_name in all_results["damascus"]:
        res = all_results["damascus"][agent_name]
        probs = res["probabilities"]  # (num_runs, num_steps, num_actions)
        avg_probs = probs.mean(axis=0)  # (num_steps, num_actions)
        for arm in range(options["num_actions"]):
            ax.plot(avg_probs[:, arm], label=f"p(action {arm})", linewidth=2)
        ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label="optimal (p=0.5)")
        ax.set_title(f"{agent_name.upper()} on Damascus")
        ax.set_xlabel("Step")
        ax.set_ylabel("Probability")
        ax.set_ylim(-0.05, 1.05)
        ax.legend()
        ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Cumulative reward comparison: UCB vs IUCB on Damascus
fig, ax = plt.subplots(figsize=(8, 4))

for agent_name in ["ucb", "iucb", "classical", "experimental2"]:
    if "damascus" in all_results and agent_name in all_results["damascus"]:
        res = all_results["damascus"][agent_name]
        cumulative = np.cumsum(res["average_reward"][0, :])
        ax.plot(cumulative, label=agent_name, linewidth=2)

optimal = all_results["damascus"][next(iter(all_results["damascus"]))]["optimal_reward"]
ax.plot(np.arange(1, options["num_steps"] + 1) * optimal,
        'k--', alpha=0.5, label=f"optimal ({optimal:.0f}/step)")

ax.set_title("Cumulative Reward on Damascus")
ax.set_xlabel("Step")
ax.set_ylabel("Cumulative Reward")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()